# 2 TLS Markovian Example

This example evolves two TLSs in a bidirectional Markovian waveguide and monitors excitation transfer and emitted field populations.

In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
import wqedmps as qmps


def two_tls_product_state(atom1_excited: bool, atom2_excited: bool) -> np.ndarray:
    """Build a local product tensor |atom1> ⊗ |atom2>."""
    atom1 = (
        np.array([0.0, 1.0], dtype=complex)
        if atom1_excited
        else np.array([1.0, 0.0], dtype=complex)
    )
    atom2 = (
        np.array([0.0, 1.0], dtype=complex)
        if atom2_excited
        else np.array([1.0, 0.0], dtype=complex)
    )
    return np.kron(atom1, atom2).reshape(1, 4, 1)


def run_two_tls_markovian_example(
    show: bool = True,
) -> tuple[qmps.InputParams, qmps.Bins, dict[str, np.ndarray | float]]:
    """Run a 2-TLS Markovian waveguide example."""
    params = qmps.InputParams(
        delta_t=0.04,
        tmax=8.0,
        d_sys_total=np.array([2, 2]),
        d_t_total=np.array([2, 2]),
        gamma_l=0.35,
        gamma_r=0.35,
        gamma_l2=0.35,
        gamma_r2=0.35,
        phase=np.pi / 2.0,
        bond_max=24,
        atol=1e-12,
    )

    initial_system = two_tls_product_state(atom1_excited=True, atom2_excited=False)
    initial_field = qmps.vacuum(params.tmax, params)
    hamiltonian = qmps.hamiltonian_2tls_mar(params)
    bins = qmps.t_evol_mar_seemps(
        hamiltonian,
        initial_system,
        initial_field,
        params,
    )

    times = np.asarray(bins.times, dtype=float)
    atom1_pop_op = np.kron(qmps.tls_pop(2), np.eye(2, dtype=complex))
    atom2_pop_op = np.kron(np.eye(2, dtype=complex), qmps.tls_pop(2))

    atom1_pop = np.asarray(
        qmps.single_time_expectation(bins.system_states, atom1_pop_op),
        dtype=float,
    )
    atom2_pop = np.asarray(
        qmps.single_time_expectation(bins.system_states, atom2_pop_op),
        dtype=float,
    )
    output_bin_pop = np.asarray(
        qmps.single_time_expectation(
            bins.output_field_states,
            [qmps.num_op_l(params.d_t_total), qmps.num_op_r(params.d_t_total)],
        ),
        dtype=float,
    )
    emitted_l = np.cumsum(output_bin_pop[0])
    emitted_r = np.cumsum(output_bin_pop[1])
    total_excitation = atom1_pop + atom2_pop + emitted_l + emitted_r

    observables = {
        "times": times,
        "atom1_pop": atom1_pop,
        "atom2_pop": atom2_pop,
        "output_bin_pop_l": output_bin_pop[0],
        "output_bin_pop_r": output_bin_pop[1],
        "emitted_l": emitted_l,
        "emitted_r": emitted_r,
        "total_excitation": total_excitation,
        "max_abs_error": float(np.max(np.abs(total_excitation - 1.0))),
    }

    if show:
        fig, axes = plt.subplots(3, 1, figsize=(8.0, 8.5), sharex=True)

        axes[0].plot(times, atom1_pop, lw=2.5, label="Atom 1 excited population")
        axes[0].plot(times, atom2_pop, lw=2.5, label="Atom 2 excited population")
        axes[0].set_ylabel("Population")
        axes[0].set_title("2 TLSs in a Markovian bidirectional waveguide")
        axes[0].legend()
        axes[0].grid(alpha=0.3)

        axes[1].plot(times, output_bin_pop[0], lw=2.5, label="Left-bin population")
        axes[1].plot(times, output_bin_pop[1], lw=2.5, label="Right-bin population")
        axes[1].set_ylabel("Local bin population")
        axes[1].legend()
        axes[1].grid(alpha=0.3)

        axes[2].plot(times, emitted_l, lw=2.5, label="Cumulative left output")
        axes[2].plot(times, emitted_r, lw=2.5, label="Cumulative right output")
        axes[2].plot(times, total_excitation, "--", lw=1.8, label="Diagnostic total")
        axes[2].axhline(1.0, color="k", linestyle="--", lw=1.2, label="1")
        axes[2].set_xlabel("Time")
        axes[2].set_ylabel("Total excitation")
        axes[2].legend()
        axes[2].grid(alpha=0.3)

        plt.tight_layout()
        plt.show()

    return params, bins, observables


run_two_tls_markovian_example()
